# Mental Health GraphRAG Pipeline — v2

**Workflow:**
1. Configure your LLM (local Ollama or cloud)
2. Load & parse markdown documents
3. Enrich nodes with context (resumable, auto-saved)
4. Extract knowledge graph triplets (resumable, auto-saved)
5. Load final checkpoints
6. Community detection (Leiden)
7. Community summarization
8. Query engine (local + global + hybrid)
9. Visualization

> **TIP — Local models (Ollama):** Steps 3 and 4 can take a long time. Each cell auto-saves progress after every node. If interrupted, just re-run the cell and it picks up where it left off.

---
## Step 0 — Configuration
Set your model here **once**. All subsequent steps will use this `llm` object.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# ─────────────────────────────────────────────────────────────────────────────
# MODEL SELECTION — uncomment ONE block below
# ─────────────────────────────────────────────────────────────────────────────

# ── OPTION A: Ollama (LOCAL) ──────────────────────────────────────────────────
# Runs entirely on your machine. No API key needed. Requires Ollama to be
# running: https://ollama.com  →  `ollama serve`
#
# Recommended models (pull first with `ollama pull <name>`):
#   Fast / light : "phi3", "gemma2:2b", "qwen2:1.5b"
#   Balanced     : "llama3", "mistral", "gemma2", "qwen2"
#   High quality : "llama3.1", "mixtral", "deepseek-r1:8b"
#
# from llama_index.llms.ollama import Ollama
# llm = Ollama(model="qwen3.5:0.8b", request_timeout=300.0)


# ── OPTION B: Ollama (CLOUD / REMOTE SERVER) ─────────────────────────────────
# Use Ollama running on a remote machine (cloud VM, server, etc.).
#
# Setup:
#   1. Set OLLAMA_CLOUD_URL in your .env  (e.g.  OLLAMA_CLOUD_URL=http://1.2.3.4:11434)
#      OR replace the fallback URL string below with your server address.
#   2. Set the model name to one that is already pulled on the remote server.
#      ⚠️  Ollama model tags are size variants like :7b, :14b, :latest — NOT ":cloud"
#
# Examples of valid model names for a remote server:
#   "llama3"          (= llama3:latest)
#   "llama3.1:8b"
#   "qwen2.5:7b"
#   "mistral:7b"
#   "deepseek-r1:8b"
#
from llama_index.llms.ollama import Ollama
llm = Ollama(
    model="glm-4.7:cloud",                                                         # ← change to your remote model
    base_url=os.getenv("OLLAMA_CLOUD_URL", "http://your-server-ip:11434"), # ← set OLLAMA_CLOUD_URL in .env
    request_timeout=300.0,
)


# ── OPTION C: OpenAI (CLOUD) ──────────────────────────────────────────────────
# Requires OPENAI_API_KEY in your .env file.
#
# Models:
#   Fast / cheap : "gpt-4o-mini"
#   High quality : "gpt-4o"
#
# from llama_index.llms.openai import OpenAI
# llm = OpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))


# ── OPTION D: Anthropic Claude (CLOUD) ───────────────────────────────────────
# Requires ANTHROPIC_API_KEY in your .env file.
#
# Models:
#   Fast / cheap : "claude-3-haiku-20240307"
#   High quality : "claude-3-5-sonnet-20241022"
#
# from llama_index.llms.anthropic import Anthropic
# llm = Anthropic(model="claude-3-haiku-20240307", api_key=os.getenv("ANTHROPIC_API_KEY"))


# ── OPTION E: Google Gemini (CLOUD) ──────────────────────────────────────────
# Requires GOOGLE_API_KEY in your .env file.
#
# Models:
#   Fast / cheap : "models/gemini-1.5-flash"
#   High quality : "models/gemini-1.5-pro"
#
# from llama_index.llms.gemini import Gemini
# llm = Gemini(model="models/gemini-1.5-flash", api_key=os.getenv("GOOGLE_API_KEY"))


# ─────────────────────────────────────────────────────────────────────────────
print(f"✅ LLM configured: {llm.__class__.__name__}")

✅ LLM configured: Ollama


In [2]:
# test the LLM with a simple prompt
response = llm.complete("What is 2+2?")
print(f"LLM response test: {response}")

LLM response test: 4


---
## Step 1 — Load & Parse Markdown Documents

In [3]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import MarkdownNodeParser

# 1. Load markdown files from the data folder
reader = SimpleDirectoryReader(input_dir="./data/markdown",required_exts=[".md"])
documents = reader.load_data()

# 2. Split by markdown headers (## / ###) — preserves logical sections
parser = MarkdownNodeParser()
nodes = parser.get_nodes_from_documents(documents)

print(f"--- Division Complete ---")
print(f"Total logical sections found: {len(nodes)}\n")

for i, node in enumerate(nodes[:3]):
    header = node.metadata.get("header_path", "Main Title")
    print(f"SECTION {i}: {header}")
    print(f"TEXT PREVIEW: {node.text[:100]}...")
    print("-" * 30)

--- Division Complete ---
Total logical sections found: 164

SECTION 0: /
TEXT PREVIEW: An illustration shows a person lying in bed, covering their face with their hands, with a scribbled ...
------------------------------
SECTION 1: /
TEXT PREVIEW: ## Is it stress or anxiety?

<table>
  <tbody>
    <tr>
        <td>Stress</td>
        <td>Bo...
------------------------------
SECTION 2: /
TEXT PREVIEW: ## Ways to Cope
* Keep a journal.
* Download an app with relaxation exercises.
* Exercise and eat...
------------------------------


---
## Step 2 — Node Enrichment (Contextual Tagging)

Each node gets a 1-sentence context tag so the graph knows which disorder and sub-topic it belongs to.

> **Progress is auto-saved** after every node into `./checkpoints/enrichment_progress.jsonl`.  
> If interrupted, re-run **2a → 2b → 2c** and it will continue from the last saved node.

### 2a — Check Progress

In [4]:
import json
import os

ENRICHMENT_PROGRESS_FILE = "./checkpoints/enrichment_progress.jsonl"
os.makedirs("./checkpoints", exist_ok=True)

# Auto-detect how many nodes are already processed
already_enriched = 0
if os.path.exists(ENRICHMENT_PROGRESS_FILE):
    with open(ENRICHMENT_PROGRESS_FILE, "r", encoding="utf-8") as f:
        already_enriched = sum(1 for line in f if line.strip())
    print(f"Progress file found: {already_enriched} / {len(nodes)} nodes already enriched.")
else:
    print("No progress file found. Will start from scratch.")

# ─────────────────────────────────────────────────────────────────────────────
# RESUME CONTROL
# Leave as None  → auto-resume from last saved position
# Set to 0       → restart from the very beginning (overwrites previous progress)
# Set to N (int) → start from node index N
# ─────────────────────────────────────────────────────────────────────────────
ENRICH_START_INDEX = None  # <── change if needed

if ENRICH_START_INDEX is None:
    ENRICH_START_INDEX = already_enriched

if ENRICH_START_INDEX == 0 and os.path.exists(ENRICHMENT_PROGRESS_FILE):
    os.remove(ENRICHMENT_PROGRESS_FILE)
    print("⚠️  Restart requested — progress file cleared.")

print(f"\n─── Enrichment will START from index : {ENRICH_START_INDEX}")
print(f"─── Nodes remaining                  : {len(nodes) - ENRICH_START_INDEX}")

Progress file found: 164 / 164 nodes already enriched.

─── Enrichment will START from index : 164
─── Nodes remaining                  : 0


### 2b — Run Enrichment Loop
Each enriched node is **immediately appended** to the progress file.

In [5]:
print(f"Starting enrichment from node {ENRICH_START_INDEX} / {len(nodes) - 1}...\n")

with open(ENRICHMENT_PROGRESS_FILE, "a", encoding="utf-8") as progress_f:
    for i in range(ENRICH_START_INDEX, len(nodes)):
        node = nodes[i]
        source_doc = node.metadata.get("file_name", "Unknown Document")

        prompt = f"""
        Below is a section from a mental health brochure titled '{source_doc}'. 
        Please provide a 1-sentence context that describes what disorder and what sub-topic 
        (e.g., Symptoms, Treatment, Causes) this text belongs to. 
        
        TEXT: {node.text[:500]}...
        
        Output only the context sentence and nothing else.
        """

        context_tag = llm.complete(prompt).text.strip()
        enriched_text = f"[CONTEXT: {context_tag}]\n\n" + node.text

        # Save this node's result immediately
        record = {"idx": i, "text": enriched_text}
        progress_f.write(json.dumps(record, ensure_ascii=False) + "\n")
        progress_f.flush()  # force write to disk right away

        print(f"[{i+1}/{len(nodes)}] Enriched: {context_tag[:70]}...")

print(f"\n✅ Enrichment loop finished. Progress saved to {ENRICHMENT_PROGRESS_FILE}")

Starting enrichment from node 164 / 163...


✅ Enrichment loop finished. Progress saved to ./checkpoints/enrichment_progress.jsonl


### 2c — Finalize Enrichment → Save to PKL
Run this **once** when all nodes are enriched. Reconstructs the nodes list and saves a final checkpoint.

In [6]:
import pickle
import json

# Load all saved enriched texts from the progress file
enriched_texts = {}
with open(ENRICHMENT_PROGRESS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            enriched_texts[record["idx"]] = record["text"]

if len(enriched_texts) < len(nodes):
    missing = len(nodes) - len(enriched_texts)
    print(f"⚠️  Warning: {missing} node(s) not yet enriched. Run Step 2b first.")
else:
    # Apply enriched texts back to a copy of the nodes list
    enriched_nodes = []
    for i, node in enumerate(nodes):
        if i in enriched_texts:
            node.text = enriched_texts[i]
        enriched_nodes.append(node)

    with open("./checkpoints/enriched_nodes.pkl", "wb") as f:
        pickle.dump(enriched_nodes, f)

    print(f"✅ All {len(enriched_nodes)} enriched nodes saved to ./checkpoints/enriched_nodes.pkl")
    print(f"   Step 2 is COMPLETE — you can skip to Step 4 next time.")

✅ All 164 enriched nodes saved to ./checkpoints/enriched_nodes.pkl
   Step 2 is COMPLETE — you can skip to Step 4 next time.


---
## Step 3 — Graph Construction (Triplet Extraction)

For each enriched node, the LLM extracts structured medical triplets  
`(Subject : Type) -[RELATION]-> (Object : Type)`.

> **Progress is auto-saved** after every node into `./checkpoints/graph_progress.jsonl`.  
> If interrupted, re-run **3a → 3b → 3c**.

### 3a — Check Progress

In [12]:
import json
import os

GRAPH_PROGRESS_FILE = "./checkpoints/graph_progress.jsonl"
os.makedirs("./checkpoints", exist_ok=True)

# Auto-detect how many nodes are already processed
already_graph_nodes = 0
pre_loaded_triplets = []

if os.path.exists(GRAPH_PROGRESS_FILE):
    with open(GRAPH_PROGRESS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                already_graph_nodes += 1
                pre_loaded_triplets.extend(record.get("triplets", []))
    print(f"Progress file found: {already_graph_nodes} / {len(enriched_nodes)} nodes processed, "
          f"{len(pre_loaded_triplets)} triplets accumulated.")
else:
    print("No graph progress file found. Will start from scratch.")

# ─────────────────────────────────────────────────────────────────────────────
# RESUME CONTROL
# Leave as None  → auto-resume from last saved position
# Set to 0       → restart from the very beginning (overwrites previous progress)
# Set to N (int) → start from node index N
# ─────────────────────────────────────────────────────────────────────────────
GRAPH_START_INDEX = None  # <── change if needed

if GRAPH_START_INDEX is None:
    GRAPH_START_INDEX = already_graph_nodes

if GRAPH_START_INDEX == 0 and os.path.exists(GRAPH_PROGRESS_FILE):
    os.remove(GRAPH_PROGRESS_FILE)
    pre_loaded_triplets = []
    print("⚠️  Restart requested — graph progress file cleared.")

print(f"\n─── Graph construction will START from index : {GRAPH_START_INDEX}")
print(f"─── Nodes remaining                          : {len(enriched_nodes) - GRAPH_START_INDEX}")

Progress file found: 89 / 164 nodes processed, 497 triplets accumulated.

─── Graph construction will START from index : 89
─── Nodes remaining                          : 75


### 3b — Run Extraction Loop
Each node's triplets are **immediately appended** to the progress file.

In [13]:
import re
from llama_index.core.llms import ChatMessage

rigid_prompt = """
You are a medical knowledge graph extractor.
Extract relationships from the text in this EXACT strict format:
(SUBJECT : TYPE) -[RELATION]-> (OBJECT : TYPE)

Allowed Types: CONDITION, SYMPTOM, TREATMENT, MEDICATION, SAFETY_RESOURCE, DEMOGRAPHIC
Allowed Relations: HAS_SYMPTOM, TREATED_BY, PRESCRIBES, SUITABLE_FOR, CONTRAINDICATED_FOR, URGENT_ACTION

Do not explain. Do not add extra text. Output only valid triplets.
If no triplets exist, output nothing.
"""

triplet_pattern = r'\(([^:]+)\s*:\s*([^)]+)\)\s*-\[([^]]+)\]->\s*\(([^:]+)\s*:\s*([^)]+)\)'

print(f"Starting graph extraction from node {GRAPH_START_INDEX} / {len(enriched_nodes) - 1}...\n")

with open(GRAPH_PROGRESS_FILE, "a", encoding="utf-8") as progress_f:
    for i in range(GRAPH_START_INDEX, len(enriched_nodes)):
        node = enriched_nodes[i]

        system_msg = ChatMessage(role="system", content=rigid_prompt)
        user_msg = ChatMessage(role="user", content=f"Text to extract from:\n{node.text}")
        response = llm.chat([system_msg, user_msg]).message.content

        node_triplets = []
        if response and response.strip() not in ("", "None"):
            matches = re.findall(triplet_pattern, response)
            for subj, subj_type, rel, obj, obj_type in matches:
                node_triplets.append({
                    "subj":      subj.strip().upper(),
                    "subj_type": subj_type.strip().upper(),
                    "rel":       rel.strip().upper(),
                    "obj":       obj.strip().upper(),
                    "obj_type":  obj_type.strip().upper(),
                })


        # Save this node's result immediately (even if 0 triplets, so index stays consistent)
        record = {"node_idx": i, "triplets": node_triplets}
        progress_f.write(json.dumps(record, ensure_ascii=False) + "\n")
        progress_f.flush()

        print(f"[{i+1}/{len(enriched_nodes)}] Node {i}: {len(node_triplets)} triplets extracted.")

print(f"\n✅ Graph extraction loop finished. Progress saved to {GRAPH_PROGRESS_FILE}")

Starting graph extraction from node 89 / 163...

[90/164] Node 89: 9 triplets extracted.
[91/164] Node 90: 6 triplets extracted.
[92/164] Node 91: 0 triplets extracted.
[93/164] Node 92: 0 triplets extracted.
[94/164] Node 93: 0 triplets extracted.
[95/164] Node 94: 0 triplets extracted.
[96/164] Node 95: 9 triplets extracted.
[97/164] Node 96: 14 triplets extracted.
[98/164] Node 97: 0 triplets extracted.
[99/164] Node 98: 30 triplets extracted.
[100/164] Node 99: 6 triplets extracted.
[101/164] Node 100: 4 triplets extracted.
[102/164] Node 101: 6 triplets extracted.
[103/164] Node 102: 7 triplets extracted.
[104/164] Node 103: 3 triplets extracted.
[105/164] Node 104: 3 triplets extracted.
[106/164] Node 105: 4 triplets extracted.
[107/164] Node 106: 2 triplets extracted.
[108/164] Node 107: 0 triplets extracted.
[109/164] Node 108: 0 triplets extracted.
[110/164] Node 109: 3 triplets extracted.
[111/164] Node 110: 8 triplets extracted.
[112/164] Node 111: 7 triplets extracted.
[113

ResponseError: Post "https://ollama.com:443/api/chat?ts=1774400097": read tcp 10.149.217.187:59621->34.36.133.15:443: wsarecv: An established connection was aborted by the software in your host machine. (status code: 500)

### 3c — Finalize Graph → Save to PKL
Run this **once** when all nodes are processed.

In [ ]:
import pickle
import json
from llama_index.core.graph_stores.types import EntityNode, Relation

# Load all triplets from the progress file
all_raw_triplets = []
nodes_processed = 0
with open(GRAPH_PROGRESS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            all_raw_triplets.extend(record.get("triplets", []))
            nodes_processed += 1

if nodes_processed < len(enriched_nodes):
    missing = len(enriched_nodes) - nodes_processed
    print(f"⚠️  Warning: {missing} node(s) not yet processed. Run Step 3b first.")
else:
    # Reconstruct EntityNode and Relation objects
    custom_entities = {}
    custom_relations = []

    for t in all_raw_triplets:
        subj_node = EntityNode(name=t["subj"], label=t["subj_type"])
        obj_node  = EntityNode(name=t["obj"],  label=t["obj_type"])
        relation  = Relation(source_id=subj_node.id, target_id=obj_node.id, label=t["rel"])

        custom_entities[subj_node.id] = subj_node
        custom_entities[obj_node.id]  = obj_node
        custom_relations.append(relation)

    with open("./checkpoints/custom_entities.pkl", "wb") as f:
        pickle.dump(custom_entities, f)
    with open("./checkpoints/custom_relations.pkl", "wb") as f:
        pickle.dump(custom_relations, f)

    print(f"✅ Graph data finalized!")
    print(f"   Entities  : {len(custom_entities)}")
    print(f"   Relations : {len(custom_relations)}")
    print(f"   Step 3 is COMPLETE — you can skip to Step 4 next time.")

---
## Step 4 — Load Checkpoints
**Start here** if Steps 2 and 3 are already complete.

In [ ]:
import pickle
import os

print("Loading checkpoints from disk...\n")

# Enriched nodes
if os.path.exists("./checkpoints/enriched_nodes.pkl"):
    with open("./checkpoints/enriched_nodes.pkl", "rb") as f:
        enriched_nodes = pickle.load(f)
    print(f"✅ Loaded {len(enriched_nodes)} enriched nodes.")
else:
    print("⚠️  enriched_nodes.pkl not found — run Steps 2a → 2b → 2c first.")

# Entities
if os.path.exists("./checkpoints/custom_entities.pkl"):
    with open("./checkpoints/custom_entities.pkl", "rb") as f:
        custom_entities = pickle.load(f)
    print(f"✅ Loaded {len(custom_entities)} entities.")
else:
    print("⚠️  custom_entities.pkl not found — run Steps 3a → 3b → 3c first.")

# Relations
if os.path.exists("./checkpoints/custom_relations.pkl"):
    with open("./checkpoints/custom_relations.pkl", "rb") as f:
        custom_relations = pickle.load(f)
    print(f"✅ Loaded {len(custom_relations)} relations.")
else:
    print("⚠️  custom_relations.pkl not found — run Steps 3a → 3b → 3c first.")

---
## Step 5 — Community Detection (Leiden Algorithm)

Partitions the knowledge graph into topic clusters using the Leiden algorithm.  
Each community represents a coherent medical topic (e.g., "Depression treatments", "Anxiety symptoms").

In [ ]:
%pip install -q "numpy<2" "graspologic>=3.4.1" "networkx>=3.0"

import importlib
import sys
from collections import defaultdict

import numpy as np
import networkx as nx

# Compatibility shim for packages that try `import numpy.char`
try:
    sys.modules.setdefault("numpy.char", importlib.import_module("numpy.core.defchararray"))
except Exception:
    sys.modules.setdefault("numpy.char", np.char)

from graspologic.partition import leiden

# 1. Build a NetworkX graph
G = nx.Graph()
for eid, entity in custom_entities.items():
    G.add_node(eid, name=entity.name, label=entity.label)
for rel in custom_relations:
    G.add_edge(rel.source_id, rel.target_id, relation=rel.label)

print(f"Graph built: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# 2. Run Leiden
partition = leiden(G, random_seed=42)

# 3. Organize into communities dict
communities = defaultdict(list)
for node_id, community_id in partition.items():
    communities[community_id].append(node_id)

print(f"\nLeiden found {len(communities)} communities:\n")

for comm_id, member_ids in sorted(communities.items(), key=lambda x: -len(x[1])):
    member_names  = [custom_entities[mid].name  for mid in member_ids if mid in custom_entities]
    member_labels = [custom_entities[mid].label for mid in member_ids if mid in custom_entities]
    community_set = set(member_ids)
    internal_rels = [r for r in custom_relations
                     if r.source_id in community_set and r.target_id in community_set]

    print(f"Community {comm_id} ({len(member_names)} entities):")
    for name, label in zip(member_names, member_labels):
        print(f"  - {name} [{label}]")
    print(f"  Internal relations: {len(internal_rels)}")
    print()

### 5b — Community Graph Visualization (Matplotlib)

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from llama_index.core.graph_stores.types import EntityNode

G_vis = nx.Graph()
for eid, entity in custom_entities.items():
    G_vis.add_node(eid, name=entity.name, label=entity.label)
for rel in custom_relations:
    if rel.source_id in custom_entities and rel.target_id in custom_entities:
        G_vis.add_edge(rel.source_id, rel.target_id, relation=rel.label)

random.seed(42)
comm_ids_sorted = sorted(communities.keys(), key=lambda c: -len(communities[c]))
cmap = plt.cm.get_cmap("tab20", max(len(communities), 20))
community_colors_mpl = {comm_id: cmap(idx % 20) for idx, comm_id in enumerate(comm_ids_sorted)}

node_to_comm = {mid: comm_id for comm_id, member_ids in communities.items() for mid in member_ids}

MARKER_MAP = {
    "CONDITION": "D", "SYMPTOM": "o", "TREATMENT": "s",
    "MEDICATION": "^", "SAFETY_RESOURCE": "*", "DEMOGRAPHIC": "p",
}

print("Computing layout (this may take a moment)...")
pos = nx.spring_layout(G_vis, k=0.5, iterations=50, seed=42)

fig, ax = plt.subplots(figsize=(20, 14))
ax.set_facecolor("#1a1a2e")
fig.patch.set_facecolor("#1a1a2e")

nx.draw_networkx_edges(G_vis, pos, ax=ax, alpha=0.15, edge_color="#555555", width=0.5)

for entity_type, marker in MARKER_MAP.items():
    type_nodes = [n for n in G_vis.nodes()
                  if custom_entities.get(n, EntityNode(name="", label="")).label == entity_type]
    if not type_nodes:
        continue
    type_pos = {n: pos[n] for n in type_nodes if n in pos}
    xs = [type_pos[n][0] for n in type_nodes if n in type_pos]
    ys = [type_pos[n][1] for n in type_nodes if n in type_pos]
    colors_filtered = [community_colors_mpl.get(node_to_comm.get(n, -1), (0.5, 0.5, 0.5, 1.0))
                       for n in type_nodes if n in type_pos]
    sizes_filtered = [max(50, G_vis.degree(n) * 15) for n in type_nodes if n in type_pos]
    ax.scatter(xs, ys, c=colors_filtered, s=sizes_filtered, marker=marker,
               edgecolors="white", linewidths=0.3, zorder=3, label=entity_type)

degree_threshold = sorted([G_vis.degree(n) for n in G_vis.nodes()], reverse=True)
cutoff = degree_threshold[min(25, len(degree_threshold) - 1)] if degree_threshold else 0
labels = {n: custom_entities[n].name for n in G_vis.nodes()
          if n in custom_entities and G_vis.degree(n) >= cutoff}
nx.draw_networkx_labels(G_vis, pos, labels, font_size=6, font_color="white",
                        font_weight="bold", ax=ax)

type_handles = [plt.Line2D([0], [0], marker=m, color='w', markerfacecolor='gray',
                            markersize=10, linestyle='None', label=t)
                for t, m in MARKER_MAP.items()]
top_n_legend = min(10, len(comm_ids_sorted))
comm_handles = [mpatches.Patch(color=community_colors_mpl[comm_ids_sorted[i]],
                               label=f"Community {comm_ids_sorted[i]} ({len(communities[comm_ids_sorted[i]])} nodes)")
                for i in range(top_n_legend)]

legend1 = ax.legend(handles=type_handles, loc="upper left", title="Entity Types",
                    fontsize=8, title_fontsize=9, facecolor="#2a2a4a",
                    edgecolor="white", labelcolor="white")
legend1.get_title().set_color("white")
ax.add_artist(legend1)
legend2 = ax.legend(handles=comm_handles, loc="lower left", title="Top Communities (Leiden)",
                    fontsize=7, title_fontsize=9, facecolor="#2a2a4a",
                    edgecolor="white", labelcolor="white")
legend2.get_title().set_color("white")

ax.set_title("Mental Health Knowledge Graph — Community Detection (Leiden Algorithm)",
             fontsize=16, color="white", fontweight="bold", pad=20)
ax.axis("off")
plt.tight_layout()
plt.savefig("community_graph_matplotlib.png", dpi=150,
            facecolor=fig.get_facecolor(), bbox_inches="tight")
plt.show()
print(f"\n✅ Saved to community_graph_matplotlib.png")
print(f"   Nodes: {G_vis.number_of_nodes()} | Edges: {G_vis.number_of_edges()} | Communities: {len(communities)}")

---
## Step 6 — Community Summarization

The LLM generates a natural-language summary for each community cluster.  
These summaries power the **global search** layer of the query engine.

In [ ]:
from llama_index.core.llms import ChatMessage

community_summaries = {}

print(f"Summarizing {len(communities)} communities...\n")

for comm_id, member_ids in sorted(communities.items(), key=lambda x: -len(x[1])):
    members = [f"{custom_entities[mid].name} [{custom_entities[mid].label}]"
               for mid in member_ids if mid in custom_entities]

    community_set = set(member_ids)
    internal_rels = [
        f"({custom_entities[r.source_id].name}) -[{r.label}]-> ({custom_entities[r.target_id].name})"
        for r in custom_relations
        if r.source_id in community_set and r.target_id in community_set
    ]

    entities_str  = "\n".join(f"  - {m}" for m in members)
    relations_str = "\n".join(f"  - {r}" for r in internal_rels) if internal_rels else "  (no internal relations)"

    system_msg = ChatMessage(
        role="system",
        content=(
            "You are a medical knowledge graph analyst. Given a cluster of entities and their "
            "relationships from mental health brochures, write a concise summary (3-5 sentences) "
            "that describes:\n"
            "1. The main disorder(s) or topic this community is about\n"
            "2. Key symptoms, treatments, or medications mentioned\n"
            "3. How the entities are related to each other\n"
            "Be factual and clinical. Do not add information not present in the data."
        )
    )
    user_msg = ChatMessage(
        role="user",
        content=(
            f"Summarize this community of medical entities:\n\n"
            f"ENTITIES:\n{entities_str}\n\n"
            f"RELATIONSHIPS:\n{relations_str}"
        )
    )

    response = llm.chat([system_msg, user_msg]).message.content.strip()
    community_summaries[comm_id] = {
        "summary":   response,
        "entities":  members,
        "relations": internal_rels,
        "size":      len(members),
    }

    print(f"Community {comm_id} ({len(members)} entities):")
    print(f"  Summary: {response[:150]}...")
    print()

print(f"\n✅ Generated summaries for {len(community_summaries)} communities.")

---
## Step 7 — Query Engine (Local + Global + Hybrid)

- **Local search**: triplet-level (specific facts)
- **Global search**: community-summary-level (broad topics)
- **Hybrid**: both combined

In [ ]:
from llama_index.core.graph_stores import SimplePropertyGraphStore
from llama_index.core import PropertyGraphIndex
from llama_index.core.llms import ChatMessage

# ── Rebuild index from saved data ────────────────────────────────────────────
print("--- REBUILDING INDEX ---")
graph_store = SimplePropertyGraphStore()
graph_store.upsert_nodes(list(custom_entities.values()))
graph_store.upsert_relations(custom_relations)

index = PropertyGraphIndex.from_existing(property_graph_store=graph_store, llm=llm)
local_query_engine = index.as_query_engine()

print(f"✅ Index ready — {len(custom_entities)} entities, {len(custom_relations)} relations")


# ── Global search function ────────────────────────────────────────────────────
def global_query(question, community_summaries, llm, top_k=5):
    question_lower = question.lower()
    scored = []
    for comm_id, data in community_summaries.items():
        combined = data["summary"].lower() + " " + " ".join(data["entities"]).lower()
        q_words = set(question_lower.split())
        score = sum(1 for w in q_words if w in combined)
        scored.append((comm_id, score, data))
    scored.sort(key=lambda x: -x[1])
    top_communities = scored[:top_k]

    context_parts = [
        f"[Community {c[0]} | {c[2]['size']} entities | Relevance: {c[1]}]\n"
        f"{c[2]['summary']}\n"
        f"Key entities: {', '.join(c[2]['entities'][:10])}"
        for c in top_communities
    ]
    context = "\n\n---\n\n".join(context_parts)

    system_msg = ChatMessage(
        role="system",
        content=(
            "You are a mental health information assistant. Answer the user's question "
            "based ONLY on the community summaries provided below. These summaries come "
            "from a knowledge graph built from NIMH mental health brochures.\n\n"
            "Be accurate, cite which community the information comes from, and if the "
            "information is not available in the summaries, say so clearly.\n\n"
            f"COMMUNITY SUMMARIES:\n{context}"
        )
    )
    user_msg = ChatMessage(role="user", content=question)
    response = llm.chat([system_msg, user_msg]).message.content.strip()
    return response, top_communities


# ── Hybrid search function ────────────────────────────────────────────────────
def hybrid_query(question, community_summaries, local_query_engine, llm):
    print(f"Question: {question}\n")

    print("Searching triplets (local)...")
    local_response = local_query_engine.query(question)

    print("Searching communities (global)...")
    global_response, top_comms = global_query(question, community_summaries, llm)

    system_msg = ChatMessage(
        role="system",
        content=(
            "You are a mental health information assistant. Synthesize the following two "
            "responses into a single, comprehensive answer. The LOCAL response comes from "
            "specific knowledge graph triplets. The GLOBAL response comes from community-level "
            "summaries. Combine them without repetition.\n\n"
            f"LOCAL SEARCH RESULT:\n{str(local_response)}\n\n"
            f"GLOBAL SEARCH RESULT:\n{global_response}"
        )
    )
    user_msg = ChatMessage(role="user", content=question)
    final = llm.chat([system_msg, user_msg]).message.content.strip()

    return {
        "answer":          final,
        "local_result":    str(local_response),
        "global_result":   global_response,
        "communities_used": [(c[0], c[1]) for c in top_comms],
    }


print("\n✅ Query functions ready: local_query_engine, global_query(), hybrid_query()")

### 7b — Run Test Queries

In [ ]:
SEP = "=" * 60

# Test 1: LOCAL (specific facts)
print(SEP)
print("TEST 1 — LOCAL SEARCH")
print(SEP)
q1 = "What medications are used to treat depression?"
print(f"Q: {q1}")
print(f"A: {local_query_engine.query(q1)}\n")

# Test 2: GLOBAL (broad topic)
print(SEP)
print("TEST 2 — GLOBAL SEARCH")
print(SEP)
q2 = "Compare the treatment approaches for anxiety and depression."
global_result, used_communities = global_query(q2, community_summaries, llm)
print(f"Q: {q2}")
print(f"A: {global_result}")
print(f"Communities used: {[(c[0], c[1]) for c in used_communities]}\n")

# Test 3: HYBRID
print(SEP)
print("TEST 3 — HYBRID SEARCH")
print(SEP)
q3 = "What are the symptoms of PTSD and how is it treated?"
result = hybrid_query(q3, community_summaries, local_query_engine, llm)
print(f"\nQ: {q3}")
print(f"A: {result['answer']}")

---
## Step 8 — Interactive Graph Visualization (Pyvis)

In [ ]:
from pyvis.network import Network
import random

random.seed(42)
community_colors = {comm_id: f"#{random.randint(0, 0xFFFFFF):06x}" for comm_id in communities.keys()}
node_to_community = {mid: comm_id for comm_id, member_ids in communities.items() for mid in member_ids}

SHAPE_MAP = {
    "CONDITION": "diamond", "SYMPTOM": "dot",   "TREATMENT": "square",
    "MEDICATION": "triangle", "SAFETY_RESOURCE": "star", "DEMOGRAPHIC": "box",
}

net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", notebook=True)
net.barnes_hut(gravity=-5000, central_gravity=0.3, spring_length=150)

for eid, entity in custom_entities.items():
    comm_id = node_to_community.get(eid, 0)
    net.add_node(
        eid,
        label=entity.name,
        title=f"{entity.name}\nType: {entity.label}\nCommunity: {comm_id}",
        color=community_colors.get(comm_id, "#888888"),
        shape=SHAPE_MAP.get(entity.label, "dot"),
        size=20,
    )

for rel in custom_relations:
    net.add_edge(rel.source_id, rel.target_id, title=rel.label, label=rel.label, color="#666666")

output_path = "mental_health_knowledge_graph.html"
net.show(output_path)

print(f"✅ Graph saved to {output_path}")
print(f"   Nodes: {len(custom_entities)} | Edges: {len(custom_relations)} | Communities: {len(communities)}")
print(f"\nLegend:")
print(f"  Diamond = CONDITION | Circle = SYMPTOM | Square = TREATMENT")
print(f"  Triangle = MEDICATION | Star = SAFETY_RESOURCE | Box = DEMOGRAPHIC")
print(f"  Colors represent different Leiden communities")